# Early Sepsis Detection — XGBoost (Patient_ID Excluded)

This notebook retrains the original 12-column XGBoost pipeline **without using `Patient_ID` as a predictive feature**. `Patient_ID` is retained only for patient-level train/test grouping.

The model uses current vital signs, `Hour`, `Age`, `Gender`, and temporal features: 6-hour rolling mean/std, hourly delta, and 3-hour trend.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)

In [ ]:
df = pd.read_csv("sepsis_dataset.csv")
print("Original shape:", df.shape)

In [ ]:
df = df.dropna().copy()
df = df.sort_values(["Patient_ID", "Hour"]).reset_index(drop=True)
df["Age"] = df["Age"].astype(int)
df["SepsisLabel"] = df["SepsisLabel"].astype(int)
df["Gender"] = df["Gender"].map({"M": 1, "F": 0})
print("After dropna:", df.shape)
print(df["SepsisLabel"].value_counts())

In [ ]:
vital_features = ["HR", "SBP", "MAP", "DBP", "Resp", "O2Sat", "Temp"]
# Patient_ID is NOT a model feature. It is used only to keep patients separated.
other_features = ["Hour", "Age", "Gender"]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df["SepsisLabel"], groups=df["Patient_ID"]))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()
print("Train patients:", train_df["Patient_ID"].nunique())
print("Test patients :", test_df["Patient_ID"].nunique())

In [ ]:
def create_time_features(data, vital_cols):
    data = data.sort_values(["Patient_ID", "Hour"]).copy()
    rolling_dict, delta_dict, trend_dict = {}, {}, {}

    for col in vital_cols:
        grp = data.groupby("Patient_ID")[col]
        rolling_dict[f"{col}_mean6h"] = (
            grp.apply(lambda x: x.rolling(6, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )
        rolling_dict[f"{col}_std6h"] = (
            grp.apply(lambda x: x.rolling(6, min_periods=1).std())
            .reset_index(level=0, drop=True)
            .fillna(0)
        )
        delta_dict[f"{col}_delta"] = grp.diff().fillna(0)
        trend_dict[f"{col}_trend3h"] = (
            grp.apply(lambda x: x.diff().rolling(3).mean())
            .reset_index(level=0, drop=True)
            .fillna(0)
        )

    engineered = pd.concat(
        [
            pd.DataFrame(rolling_dict, index=data.index),
            pd.DataFrame(delta_dict, index=data.index),
            pd.DataFrame(trend_dict, index=data.index),
        ], axis=1,
    )
    data = pd.concat([data, engineered], axis=1)
    new_features = list(rolling_dict) + list(delta_dict) + list(trend_dict)
    return data, new_features

train_df, engineered_features = create_time_features(train_df, vital_features)
test_df, _ = create_time_features(test_df, vital_features)
feature_columns = vital_features + other_features + engineered_features
print("Number of model features:", len(feature_columns))
print(feature_columns)

In [ ]:
X_train = train_df[feature_columns]
y_train = train_df["SepsisLabel"]
X_test = test_df[feature_columns]
y_test = test_df["SepsisLabel"]

model = xgb.XGBClassifier(
    max_depth=3,
    min_child_weight=10,
    subsample=0.5,
    colsample_bytree=0.5,
    learning_rate=0.05,
    n_estimators=200,
    reg_alpha=10,
    reg_lambda=20,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob),
    "AUPRC": average_precision_score(y_test, y_prob),
}

for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.title("Confusion Matrix — Patient_ID Excluded")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure()
plt.plot(fpr, tpr, label=f"ROC AUC = {metrics['ROC_AUC']:.4f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob)
plt.figure()
plt.plot(recall, precision, label=f"AUPRC = {metrics['AUPRC']:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend()
plt.show()

## Save the trained model for deployment

XGBoost's JSON model format can be loaded later by XGBoost. The deployment service must recreate the same 38 features in the same order before prediction.

In [ ]:
import json

model.save_model("xgb_sepsis_no_patient_id.json")

feature_config = {
    "features": feature_columns,
    "vital_features": vital_features,
    "other_features": other_features,
    "thresholds": {"yellow": 0.30, "red": 0.70}
}

with open("model_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

print("Saved xgb_sepsis_no_patient_id.json and model_config.json")